# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Chargement des données
Les données `.dat` sont chargées nativement par Pyomo via `model.create_instance(...)` dans la section du modèle.

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = AbstractModel()

## 🔹 Sets

In [ ]:
model.PERIODES = Set()

## 🔹 Parameters

In [ ]:
model.CoutAchat = Param(model.PERIODES, within=NonNegativeReals)
model.CoutProd = Param(model.PERIODES, within=NonNegativeReals)
model.Mini = Param(model.PERIODES, within=NonNegativeReals)
model.Maxi = Param(model.PERIODES, within=NonNegativeReals)
model.CapProd = Param(model.PERIODES, within=NonNegativeReals)
model.CapBle = Param(within=NonNegativeReals)
model.CapSpag = Param(within=NonNegativeReals)
model.CoutStockBle = Param(within=NonNegativeReals)
model.CoutStockSpag = Param(within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.Achat_ble = Var(model.PERIODES, domain=NonNegativeReals)
model.Prod_spag = Var(model.PERIODES, domain=NonNegativeReals)
model.Stock_ble = Var(model.PERIODES, domain=NonNegativeReals)
model.STOCK_SPAG = Var(model.PERIODES, domain=NonNegativeReals)

## 🔹 Data

In [ ]:
model = model.create_instance('../data/Pastissimo_pipeline_clean_data.dat')

## 🔹 Constraints

In [ ]:
model.c0 = Constraint(expr=model.Stock_ble[1] == 2 + model.Achat_ble[1] - model.Prod_spag[1])
model.c1 = Constraint(expr=model.Stock_ble[6] == 2)
model.c2 = Constraint(expr=model.STOCK_SPAG[1] == model.Prod_spag[1] - 4)
model.c3 = Constraint(expr=model.STOCK_SPAG[6] == 0)
model.c_for_0 = ConstraintList()
for p in model.PERIODES:
    model.c_for_0.add(model.Achat_ble[p] >= model.Mini[p])
model.c_for_1 = ConstraintList()
for p in model.PERIODES:
    model.c_for_1.add(model.Achat_ble[p] <= model.Maxi[p])
model.c_for_2 = ConstraintList()
for p in model.PERIODES:
    model.c_for_2.add(model.Prod_spag[p] <= model.CapProd[p])
model.c_for_3 = ConstraintList()
for p in model.PERIODES:
    model.c_for_3.add(model.Stock_ble[p] <= model.CapBle)
model.c_for_4 = ConstraintList()
for p in model.PERIODES:
    model.c_for_4.add(model.STOCK_SPAG[p] <= model.CapSpag)
model.c_for_5 = ConstraintList()
for i in model.PERIODES:
    if i >= 2:
        model.c_for_5.add(model.Stock_ble[i] == model.Stock_ble[i-1] + model.Achat_ble[i] - model.Prod_spag[i])
model.c_for_6 = ConstraintList()
for i in model.PERIODES:
    if i >= 2:
        model.c_for_6.add(model.STOCK_SPAG[i] == model.STOCK_SPAG[i-1] + model.Prod_spag[i] - 4)

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.CoutAchat[p] * model.Achat_ble[p] + model.CoutProd[p] * model.Prod_spag[p] + model.CoutStockBle * model.Stock_ble[p] + model.CoutStockSpag * model.STOCK_SPAG[p] for p in model.PERIODES), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')